# Statistical Methods Interview Drills (nb03)

Scenario-grounded statistical-methods practice, built like nb01/nb02: **pick a problem → diagnose → implement**. Topics: A/B testing, power & sample size, hypothesis tests, regression, claims metrics (PDC/PMPM).

**Every code cell collapses** (▼ Show code / ▲ Hide code) so you can focus on the prompt and your work.

**Flow:**
1. **Pick a problem** — choose Topic, Scenario (same industries as nb01/nb02), Difficulty, and Source (new / replay a solved one), then **Generate**.
2. **Diagnose** — *before* coding, work out the approach: **Walkthrough** (guided dropdowns that lead you to the right test) or **Solve** (write your diagnosis, get feedback).
3. **Implement** — the scenario is shown above a runnable Python editor: **Test** to run, **Submit** to auto-check your numbers + get a Claude rubric, **Reveal reference** for the model solution.

Run the Setup cell first.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Setup ──
import os, sys, io, contextlib, traceback, math
from pathlib import Path
import numpy as np
from scipy import stats
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
try:
    from dotenv import load_dotenv
    load_dotenv(Path(os.getcwd()) / ".env"); load_dotenv(Path(os.getcwd()).parent / ".env")
except Exception: pass
sys.path.insert(0, os.getcwd())
import importlib, stats_drill_utils as S; importlib.reload(S)
display(HTML("<style>.cw-mono textarea{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:13px;}</style>"))
_state = {"problem": None}
print("Ready. Topics:", ", ".join(S.subtopic_keys()))
print("Claude grading:", "ON" if S.claude_available() else "OFF (set ANTHROPIC_API_KEY in .env to enable rubric + diagnosis feedback)")

## 1. Pick a problem
Choose topic, scenario, difficulty, and source — then generate a scenario-grounded problem.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

_topic = widgets.Dropdown(options=[(S.subtopic_label(k),k) for k in S.subtopic_keys()],
                          description="Topic:", layout=widgets.Layout(width="460px"))
_scen  = widgets.Dropdown(options=S.scenario_options(), description="Scenario:", layout=widgets.Layout(width="460px"))
_diff  = widgets.ToggleButtons(options=S.DIFFICULTIES, value="moderate", description="Difficulty:")
_src   = widgets.ToggleButtons(options=[("New (generate)","new"),("Solved (replay)","solved")], value="new", description="Source:")
_replay = widgets.Dropdown(options=[("— generate a few first —","")], description="Replay:", layout=widgets.Layout(width="640px"))
_gen   = widgets.Button(description="Generate Problem", button_style="primary", layout=widgets.Layout(width="200px"))
_p_out = widgets.Output()

def _refresh_replay(*_):
    opts = S.list_problems(_topic.value)
    _replay.options = opts if opts else [("— none saved yet —","")]
def _on_src(*_):
    _replay.layout.display = "" if _src.value=="solved" else "none"
_on_src(); _src.observe(_on_src, names="value"); _topic.observe(_refresh_replay, names="value")

def _render_problem(p):
    _state["problem"] = p
    with _p_out:
        clear_output(); display(HTML(S.problem_card_html(p)))
        print("\n→ Go to Section 2 to diagnose, then Section 3 to implement (click 'Load current problem' in each).")

def _on_gen(_):
    if _src.value=="solved" and _replay.value:
        p = S.load_problem(_replay.value)
    else:
        p = S.generate_problem(_topic.value, _diff.value, _scen.value)
        S.save_problem(p)
    _render_problem(p); _refresh_replay()
_gen.on_click(_on_gen); _refresh_replay()
display(widgets.VBox([widgets.HBox([_topic,_scen]), _diff, _src, _replay,
                      _gen]), _p_out)
_on_gen(None)

## 2. Diagnose before coding
Work out the **approach** before writing code. Click **Load current problem**, then use **Walkthrough** (guided selections lead you to the right test) or **Solve** (write your own diagnosis and get feedback). Expand the references when stuck.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

_d_state = {"problem": None}
_d_load = widgets.Button(description="↻ Load current problem", button_style="info")
_d_mode = widgets.ToggleButtons(options=[("🧩 Walkthrough (guided)","wt"),("✍ Solve (write it)","solve")], value="wt")
_d_area = widgets.VBox([]); _d_head = widgets.Output()

# references (collapsible)
def _ref_accordion(p):
    which = widgets.HTML(S.worked_example_html(p["subtopic"]))
    acc = widgets.Accordion(children=[which]); acc.set_title(0, "📖 Worked example for this topic")
    acc.selected_index = None
    return acc

def _build_walkthrough(p):
    spec = S.diagnose_spec(p); dds=[]; rows=[]
    for i,st in enumerate(spec):
        dd = widgets.Dropdown(options=[("— pick —",-1)]+[(o,j) for j,o in enumerate(st["options"])],
                              value=-1, description=f"Q{i+1}:", layout=widgets.Layout(width="640px"),
                              style={"description_width":"40px"})
        dds.append(dd)
        rows.append(widgets.VBox([widgets.HTML(f"<b>{st['q']}</b>"), dd]))
    chk = widgets.Button(description="Check approach", button_style="success")
    out = widgets.Output()
    def _check(_):
        with out:
            clear_output(); allok=True; html_rows=""
            for st,dd in zip(spec,dds):
                ok = dd.value==st["correct"]; allok=allok and ok
                col="#057a55" if ok else "#b00"; mark="✔" if ok else "✗"
                pick = "—" if dd.value==-1 else st["options"][dd.value]
                html_rows += f"<div style='margin:3px 0'><b style='color:{col}'>{mark}</b> {st['q']} <i>you: {pick}</i> — {st['why']}</div>"
            display(HTML(html_rows))
            if allok:
                display(HTML("<div style='margin-top:8px;padding:8px 10px;border-left:3px solid #057a55;background:#eaf6ef'>"
                             "<b>✅ Your path lands on the right approach:</b><br>"+S.strategy_summary(p)+"</div>"))
            else:
                display(HTML("<div style='margin-top:8px;color:#b00'>Some steps are off — re-pick and check again. (The hints above point the way.)</div>"))
    chk.on_click(_check)
    return widgets.VBox(rows+[chk,out])

def _build_solve(p):
    ta = widgets.Textarea(placeholder="State: question type, the test you'd use, H0/H1, key assumptions, and what you'll report.",
                          layout=widgets.Layout(width="100%", height="140px"))
    btn = widgets.Button(description="Get feedback", button_style="success"); out = widgets.Output()
    def _fb(_):
        with out: clear_output(); display(HTML(S.claude_grade_diagnosis(p, ta.value)))
    btn.on_click(_fb)
    return widgets.VBox([ta, btn, out])

def _rebuild(*_):
    p=_d_state["problem"]
    if not p: _d_area.children=[]; return
    body = _build_walkthrough(p) if _d_mode.value=="wt" else _build_solve(p)
    _d_area.children=[_ref_accordion(p), body]
def _on_load(_):
    p=_state.get("problem")
    with _d_head:
        clear_output()
        if not p: print("Generate a problem in Section 1 first."); return
        display(HTML(S.problem_card_html(p, compact=True)))
    _d_state["problem"]=p; _rebuild()
_d_load.on_click(_on_load); _d_mode.observe(_rebuild, names="value")
display(_d_load, _d_head, _d_mode, _d_area)

## 3. Implement
The scenario is shown above the editor. Write Python (the variable `data` holds the inputs; `np`, `stats`, `math` are available). **Test** runs your code and shows output. **Submit** runs it, auto-checks your `answers` dict, and adds a Claude rubric. **Reveal reference** shows the model solution.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

_i_state = {"problem": None, "run_output": "", "answers": None}
_i_load = widgets.Button(description="↻ Load current problem", button_style="info")
_i_card = widgets.Output()
_editor = widgets.Textarea(layout=widgets.Layout(width="100%", height="220px")); _editor.add_class("cw-mono")
_test = widgets.Button(description="▶ Test (run, see output)")
_submit = widgets.Button(description="✔ Submit (check + grade)", button_style="success")
_reveal = widgets.Button(description="Reveal reference", button_style="warning")
_run_out = widgets.Output(); _grade_out = widgets.Output()

def _exec():
    p=_i_state["problem"]; g={"__builtins__":__builtins__,"np":np,"stats":stats,"math":math}; ns={"data":p["data"]}
    buf=io.StringIO()
    try:
        with contextlib.redirect_stdout(buf): exec(_editor.value, g, ns)
        _i_state["run_output"]=buf.getvalue(); _i_state["answers"]=ns.get("answers")
        return buf.getvalue(), None
    except Exception:
        return buf.getvalue(), traceback.format_exc()

def _on_iload(_):
    p=_state.get("problem")
    with _i_card:
        clear_output()
        if not p: print("Generate a problem in Section 1 first."); return
        display(HTML(S.problem_card_html(p)))
    if p:
        _i_state["problem"]=p; _editor.value=S.starter_code(p)
        with _run_out: clear_output()
        with _grade_out: clear_output()

def _on_test(_):
    if not _i_state["problem"]:
        with _run_out: clear_output(); print("Load a problem first."); return
    out,err=_exec()
    with _run_out:
        clear_output(); print(("ERROR:\n"+err) if err else (out or "(ran; set `answers` and print to see output)"))

def _gtable(rows):
    h="<table style='border-collapse:collapse'><tr><th style='text-align:left;padding:4px 10px'>Field</th><th style='padding:4px 10px'>Your value</th><th style='padding:4px 10px'>Expected</th><th style='padding:4px 10px'>Result</th></tr>"
    for r in rows:
        col="#057a55" if r["pass"] else "#b00"; mark="✔" if r["pass"] else "✗"
        h+=f"<tr><td style='padding:4px 10px'>{r['field']}</td><td style='padding:4px 10px'>{r['your']}</td><td style='padding:4px 10px'>{r['expected']}</td><td style='padding:4px 10px;color:{col}'><b>{mark}</b></td></tr>"
    return h+"</table>"

def _on_submit(_):
    if not _i_state["problem"]:
        with _grade_out: clear_output(); print("Load a problem first."); return
    out,err=_exec()
    with _grade_out:
        clear_output()
        if err: print("Your code errored:\n"+err); return
        ok,rows=S.check_numeric(_i_state["problem"], _i_state["answers"])
        verdict="<b style='color:#057a55'>All numeric checks passed.</b>" if ok else "<b style='color:#b00'>Some values are off — see table.</b>"
        display(HTML(verdict+"<br>"+_gtable(rows)))
        display(HTML(S.claude_rubric(_i_state["problem"], _editor.value, _i_state["run_output"])))

def _on_reveal(_):
    p=_i_state["problem"]
    if not p:
        with _grade_out: clear_output(); print("Load a problem first."); return
    ref=S.reference(p)
    with _grade_out:
        clear_output()
        display(HTML(f"<b>Approach:</b> {ref['approach']}"))
        display(HTML(f"<pre style='background:#0d1117;color:#c9d1d9;padding:10px;border-radius:6px;overflow:auto'>{__import__('html').escape(ref['solution_code'])}</pre>"))

_i_load.on_click(_on_iload); _test.on_click(_on_test); _submit.on_click(_on_submit); _reveal.on_click(_on_reveal)
display(_i_load, _i_card, _editor, widgets.HBox([_test,_submit,_reveal]), _run_out, _grade_out)

---
### Notes
- **Section flow:** generate in 1 → click **Load current problem** in 2 and 3 to pull it in.
- **Auto-run check** compares your `answers` dict to the reference within tolerance.
- **Claude rubric / diagnosis feedback** needs `ANTHROPIC_API_KEY` in the project `.env` (model via `DRILL_MODEL`, default `claude-sonnet-4-6`). Without it, you still get the numeric check and the reference approach.
- Generated problems are saved to `data/outputs/stats_problems/` and replayable via **Source → Solved**.
- Engine: `stats_drill_utils.py` — add topics, scenarios, difficulty rules, or diagnose steps there; the notebook stays thin.